In [1]:
import pandas as pd
from sklearn.model_selection import GridSearchCV , train_test_split 
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report 
from sklearn.preprocessing import OneHotEncoder , OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
df=pd.read_csv("loan_data.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  object 
 2   person_education                45000 non-null  object 
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  object 
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  object 
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  

In [3]:
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [4]:
x=df.drop(columns='loan_status')
y=df.loan_status

In [5]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [6]:
num_cols = x.select_dtypes(include='number').columns
obj_cols= x.select_dtypes(include='object').columns

In [7]:
x[obj_cols].nunique()

person_gender                     2
person_education                  5
person_home_ownership             4
loan_intent                       6
previous_loan_defaults_on_file    2
dtype: int64

In [8]:
x['person_education'].unique()

array(['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate'],
      dtype=object)

In [9]:
order=(' Master','High School','Bachelor','Associate','Doctorate')

In [30]:
preprocessing = ColumnTransformer(
    transformers=[
        ('oneHot_encoder', OneHotEncoder(handle_unknown='ignore'),obj_cols.drop('person_education')),
        ('ordinal_encoder',OrdinalEncoder(categories=[order],handle_unknown='use_encoded_value',unknown_value=-1),['person_education'])

    ],remainder='passthrough'
)

main_pipeline = Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('model', DecisionTreeClassifier(random_state=42))
    ]
)
grid_search_cv = GridSearchCV(
    estimator=main_pipeline,
    param_grid={
        'model__criterion':['gini','entropy'],
        'model__max_depth':[None,5,10],
        'model__min_samples_split':[2,5],
        'model__min_samples_leaf':[1,3],
        'model__splitter':['best','random']
    } ,verbose =10,n_jobs=-1
)
grid_search_cv.fit(xtrain , ytrain)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


c:\Users\AVANIY\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\compose\_column_transformer.py:1651: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


GridSearchCV(estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('oneHot_encoder',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         Index(['person_gender', 'person_home_ownership', 'loan_intent',
       'previous_loan_defaults_on_file'],
      dtype='object')),
                                                                        ('ordinal_encoder',
                                                                         OrdinalEncoder(categories=[(' '
                                                                                                     'Master',
                                                                                                     'High '
                                                                                                     'Scho...
                                                                                                     'Associate',
                                                                                                     'Doctorate')],
                                                                                        handle_unknown='use_encoded_value',
                                                                                        unknown_value=-1),
                                                                         ['person_education'])])),
                                       ('model',
                                        DecisionTreeClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__criterion': ['gini', 'entropy'],
                         'model__max_depth': [None, 5, 10],
                         'model__min_samples_leaf': [1, 3],
                         'model__min_samples_split': [2, 5],
                         'model__splitter': ['best', 'random']},
             verbose=10)

In [20]:
grid_search_cv.best_estimator_

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('oneHot_encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['person_gender', 'person_home_ownership', 'loan_intent',
       'previous_loan_defaults_on_file'],
      dtype='object')),
                                                 ('ordinal_encoder',
                                                  OrdinalEncoder(categories=[(' '
                                                                              'Master',
                                                                              'High '
                                                                              'School',
                                                                              'Bachelor',
                                                                              'Associate',
                                                                              'Doctorate')],
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['person_education'])])),
                ('model',
                 DecisionTreeClassifier(criterion='entropy', max_depth=10,
                                        random_state=42))])

In [21]:
grid_search_cv.best_params_

{'model__criterion': 'entropy',
 'model__max_depth': 10,
 'model__min_samples_leaf': 1,
 'model__min_samples_split': 2,
 'model__splitter': 'best'}

In [22]:
grid_search_cv.cv_results_

{'mean_fit_time': array([0.218188  , 0.11172085, 0.21290722, 0.08941426, 0.1913856 ,
        0.08753257, 0.19129176, 0.08468251, 0.10554581, 0.0633975 ,
        0.10707068, 0.06378617, 0.10638156, 0.06617885, 0.10939183,
        0.06296072, 0.14456534, 0.0729588 , 0.14360995, 0.07627525,
        0.14580455, 0.07474957, 0.14841175, 0.07140746, 0.22511864,
        0.09463243, 0.22018971, 0.09931488, 0.23127737, 0.09014072,
        0.22427278, 0.09153123, 0.122153  , 0.06687436, 0.11700892,
        0.06613526, 0.12384644, 0.0710371 , 0.11561675, 0.0638526 ,
        0.15837364, 0.07248678, 0.15713415, 0.07255926, 0.15281129,
        0.07119222, 0.15309892, 0.07222915]),
 'std_fit_time': array([0.00868262, 0.00734868, 0.00885035, 0.00220512, 0.00097971,
        0.0025586 , 0.00260004, 0.00144764, 0.00074858, 0.00138854,
        0.00216497, 0.00194698, 0.00127198, 0.00320316, 0.00442146,
        0.00111814, 0.00325104, 0.00224329, 0.0011377 , 0.00946266,
        0.00246176, 0.00524244, 0.005

In [23]:
results = pd.DataFrame(grid_search_cv.cv_results_)

In [28]:
results.sort_values(by='rank_test_score').head()


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
40,0.158374,0.009044,0.013265,0.000346,entropy,10,1,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.919583,0.926111,0.914861,0.920833,0.920556,0.920389,0.003583,1
42,0.157134,0.009595,0.013258,0.000354,entropy,10,1,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.919583,0.925833,0.914861,0.920556,0.920694,0.920306,0.003491,2
46,0.153099,0.001854,0.013455,0.000427,entropy,10,3,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.919167,0.926389,0.915417,0.920139,0.920000,0.920222,0.003530,3
44,0.152811,0.001778,0.013450,0.000328,entropy,10,3,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.919167,0.926389,0.915417,0.920139,0.920000,0.920222,0.003530,3
20,0.145805,0.002462,0.013553,0.000758,gini,10,3,2,best,"{'model__criterion': 'gini', 'model__max_depth...",0.916528,0.924028,0.915556,0.917778,0.920139,0.918806,0.003028,5


In [ ]:
#try grid seach cv for logistic regression and random forest- assignment